#1. Neural News Recommendation with Multi-Head Self Attention Model (NRMS)

NRMS models user interest by learning semantic representations of news articles and capturing relationships among previously clicked news in user history via multi-head self-attention. The model consists of a news encoder, a user encoder, and an MLP-based prediction layer.

The model comprises the following components:
- **`NewsEncoder`**: The news encoder comprises two submodules, whose outputs are fused through a fully connected layer to produce a unified news representation vector, which is included inside MLP used in Model 2.
    - `TextNewsEncoder`: This component processes textual features derived from concatenated title_abstract BERT embeddings, together with categorical features such as subcluster categories, which are encoded via learnable embeddings
    - `EntityNewsEncoder`: This component incorporates additional semantic knowledge by leveraging on entity-based representations, such as knowledge graph and named entity embeddings extracted from news content

- **`UserEncoder`**: The user encoder models user preferences by aggregating representations of previously clicked news articles. Each clicked news article is first encoded using the shared NewsEncoder to obtain dense vector representations. These representations are then further processed to capture relationships among the clicked news and produce a unified user representation

   - **Self-Attention Block**: This component applies multi-head self-attention to model interactions among clicked news articles, enabling the encoder to capture contextual relationships within the users' browsing history, which is included inside MLP used in Model 2
   - **Additive Attention**: This component aggregates the attended news representations into a single user vector by assigning higher weights to more relevant or informative clicked articles
   - Default User Embedding: For users without an available history (cold start users), a learnable default embedding is used to represent user preferences.
- **MLP-based Prediction layer**: The final user representation and candidate news representation are passed into the MLP alongside other User Features, News Features and context features. This component is based on Model 2, but incorporates attention.

##1.1 Load Packages

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import os
import ast
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


##1.2 Load Processed News Dataset

In [ ]:
# Load entity embeddings
news_ent_path = '/content/drive/MyDrive/data/news/news_entities.parquet'
news_entities = pd.read_parquet(news_ent_path)

In [ ]:
news_entities.head()

,news_id,kg_title_emb,kg_abstract_emb,kg_title_flag,kg_abstract_flag,ner_title_emb,ner_abstract_emb,ner_title_flag,ner_abstract_flag
0,N55528,"[0.008122858, -0.07991525, -0.016764905, 0.158...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1,0,"[-0.025332777, 0.07448415, 0.046060078, 0.0411...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1,0
1,N19639,"[-0.029497119, -0.021168852, 0.03713986, -0.11...","[-0.029497119, -0.021168852, 0.03713986, -0.11...",1,1,"[-0.007022121, 0.08226046, -0.035310447, 0.089...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1,0
2,N61837,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-0.06553931, -0.08845358, -0.015253108, -0.03...",0,1,"[-0.07378371, 0.10624181, 0.00063977536, 0.055...","[0.016665561, 0.026591385, -0.07544741, 0.0324...",1,1
3,N53526,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0061505293, -0.10125916, -0.060772542, 0.04...",0,1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-0.008782822, 0.051812273, -0.033509996, -0.0...",0,1
4,N38324,"[0.045086294, 0.05803315, 0.016441302, 0.00560...","[-0.02720732, -0.0005318837, 0.08588548, -0.13...",1,1,"[-0.0391243, 0.029080106, 0.014465057, 0.06514...","[-0.0037145745, -0.02635876, -0.027976032, 0.0...",1,1


In [ ]:
# Load news features dataset
news_features_path = '/content/drive/MyDrive/features/news_features/news_features.parquet'
news_features = pd.read_parquet(news_features_path)

In [ ]:
news_features.head()

,news_id,subclustered_category,bert_emb_0,bert_emb_1,bert_emb_2,bert_emb_3,bert_emb_4,bert_emb_5,bert_emb_6,bert_emb_7,...,topic_41,topic_42,topic_43,topic_44,topic_45,topic_46,topic_47,topic_48,topic_49,ctr_norm
0,26629,lifestyleroyals,-0.006567,0.030011,0.041711,0.008560,0.023631,0.013891,0.019353,-0.043506,...,0.000317,0.004183,-0.003760,0.011950,0.000253,-0.007036,-0.005527,0.003258,0.003991,-0.153225
1,45241,weightloss,0.013164,0.027189,0.018828,0.059265,-0.004955,-0.021681,0.014436,0.000486,...,-0.012257,0.007741,0.002588,0.003268,0.035718,0.015612,0.039606,0.018964,-0.044307,-0.153225
2,49591,CounterTerrorism_MiddleEastConflict,-0.018924,0.047626,0.033783,0.046084,0.001255,-0.003395,-0.018784,0.022281,...,0.000855,-0.005659,-0.012527,0.017737,0.015317,0.026903,-0.008147,0.028906,-0.035933,-0.153225
3,28765,health_general,0.047500,0.066250,0.007197,0.021490,-0.008358,0.053810,0.050125,0.025326,...,0.010095,-0.017281,0.006360,-0.008582,-0.017680,-0.003062,-0.004098,0.015898,0.013547,-0.153225
4,28005,medical,0.000916,0.073814,-0.011449,0.011641,0.016896,-0.063047,0.051980,0.021564,...,0.006993,0.005708,0.006243,0.000269,0.003198,0.005175,0.004035,-0.003169,0.005123,-0.153225


In [ ]:
news_features.columns.tolist()

['news_id',
 'subclustered_category',
 'bert_emb_0',
 'bert_emb_1',
 'bert_emb_2',
 'bert_emb_3',
 'bert_emb_4',
 'bert_emb_5',
 'bert_emb_6',
 'bert_emb_7',
 'bert_emb_8',
 'bert_emb_9',
 'bert_emb_10',
 'bert_emb_11',
 'bert_emb_12',
 'bert_emb_13',
 'bert_emb_14',
 'bert_emb_15',
 'bert_emb_16',
 'bert_emb_17',
 'bert_emb_18',
 'bert_emb_19',
 'bert_emb_20',
 'bert_emb_21',
 'bert_emb_22',
 'bert_emb_23',
 'bert_emb_24',
 'bert_emb_25',
 'bert_emb_26',
 'bert_emb_27',
 'bert_emb_28',
 'bert_emb_29',
 'bert_emb_30',
 'bert_emb_31',
 'bert_emb_32',
 'bert_emb_33',
 'bert_emb_34',
 'bert_emb_35',
 'bert_emb_36',
 'bert_emb_37',
 'bert_emb_38',
 'bert_emb_39',
 'bert_emb_40',
 'bert_emb_41',
 'bert_emb_42',
 'bert_emb_43',
 'bert_emb_44',
 'bert_emb_45',
 'bert_emb_46',
 'bert_emb_47',
 'bert_emb_48',
 'bert_emb_49',
 'bert_emb_50',
 'bert_emb_51',
 'bert_emb_52',
 'bert_emb_53',
 'bert_emb_54',
 'bert_emb_55',
 'bert_emb_56',
 'bert_emb_57',
 'bert_emb_58',
 'bert_emb_59',
 'bert_emb_6

##1.3 Load in User Dataset

In [ ]:
train = pd.read_parquet('/content/drive/MyDrive/data/train.parquet')
val= pd.read_parquet('/content/drive/MyDrive/data/val.parquet')
test = pd.read_parquet('/content/drive/MyDrive/data/test.parquet')

In [ ]:
# Load user features datasets
user_path = '/content/drive/MyDrive/features/user_features'
train_user_features = pd.read_parquet(f'{user_path}/train_user_features.parquet')
val_user_features = pd.read_parquet(f'{user_path}/val_user_features.parquet')
test_user_features = pd.read_parquet(f'{user_path}/test_user_features.parquet')

##1.4 Load in Interactions Dataset

In [ ]:
train_interaction_path = '/content/drive/MyDrive/features/interaction_features/train_interactions.parquet'
train_interactions = pd.read_parquet(train_interaction_path)
val_interaction_path = '/content/drive/MyDrive/features/interaction_features/val_interactions.parquet'
val_interactions = pd.read_parquet(val_interaction_path)
test_interaction_path = '/content/drive/MyDrive/features/interaction_features/test_interactions.parquet'
test_interactions = pd.read_parquet(test_interaction_path)

##1.5 Load in Context Dataset

In [ ]:
context_path = '/content/drive/MyDrive/features/context_features'
train_context_features = pd.read_parquet(f'{context_path}/train_context.parquet')
val_context_features = pd.read_parquet(f'{context_path}/val_context.parquet')
test_context_features = pd.read_parquet(f'{context_path}/test_context.parquet')

#2. News Encoder

News Encoder consists of two parallel news encoding branches:
1.   Text News Encoder captures semantic information from the title and abstract embeddings
2.   Entity News Encoder captures knowledge-level information from entity embeddings derived from a knowledge graph

The outputs of the Text News Encoder and Entity News Encoder are then concatenated and passed through a fusion layer to produce a unified news representation.


##2.1 Text News Encoder

`TextNewsEncoder` combines two sources of information for each new articles
1. A dense BERT emebdding derived from the concatenated title and abstract
2. A learnable embedding representing each subcluster category

Both vectors are projected into a shared 128-dimensional space and treated as a two-token sequence, where a multi-head self-attentioin layer models interactions between semantic content and categorical structure.

The contextualised tokens are then pooled and passed through a feedforward layer to produce the final 128-dimensional text representation.

In [ ]:
#Make the title_abs_matrix
bert_cols = [col for col in news_features.columns if col.startswith("bert_emb_")]
title_abs_matrix = np.stack(news_features[bert_cols].values)
title_abs_embedding_tensor = torch.from_numpy(title_abs_matrix)

In [ ]:
title_abs_matrix.shape

(51282, 768)

In [ ]:
# 1. Get unique non-null clusters
unique_clusters = news_features["subclustered_category"].dropna().unique()

# 2. Build mapping
cluster2id = {c: i for i, c in enumerate(unique_clusters)}

# 3. Add unknown cluster id
unknown_cluster_id = len(cluster2id)

# 4. Map + fill missing
news_features["subcluster_id"] = (
    news_features["subclustered_category"]
    .map(cluster2id)
    .fillna(unknown_cluster_id)
    .astype(int)
)

# 5. Convert to tensor
subcluster_id_tensor = torch.tensor(
    news_features["subcluster_id"].values,
    dtype=torch.long
)

subcluster_id_tensor.shape

torch.Size([51282])

In [ ]:
num_subclusters = len(unique_clusters) + 1
print(f"Number of subclusters: {num_subclusters}")

Number of subclusters: 96


In [ ]:
class TextNewsEncoder(nn.Module):
    def __init__(self, input_dim, num_subclusters, model_dim=128, num_heads=2, dropout=0.3):
        super().__init__()

        self.subcluster_embeddings = nn.Embedding(num_subclusters, 32)
        self.subcluster_proj = nn.Linear(32, model_dim) # New projection for subcluster embeddings

        # Project main text embedding to same dimension (d_model = 128)
        self.proj = nn.Linear(input_dim, model_dim)

        # Multi-head attention (batch_first=True is important)
        self.attention = nn.MultiheadAttention(
            embed_dim=model_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        # Feedforward layer to improve representation
        self.ffn = nn.Sequential(
            nn.Linear(model_dim, model_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

    def forward(self, title_abs_emb, subcluster_ids):
        """
        Inputs:
            title_abs_emb:   [batch, input_dim] (e.g., BERT embedding)
            subcluster_ids:  [batch] (integer IDs for subclusters)

        Output:
            text_vec: [batch, 128]
        """
        # 1. Project title_abs_emb to model_dim
        t = self.proj(title_abs_emb)      # [batch, 128]

        # 2. Look up subcluster embedding and project to model_dim
        s_emb_32 = self.subcluster_embeddings(subcluster_ids) # [batch, 32]

        s = self.subcluster_proj(s_emb_32)                     # [batch, 128]

        # 3. Stack into a 2-token sequence
        x = torch.stack([t, s], dim=1)    # [batch, 2, 128]

        # 4. Self-attention
        attn_output, _ = self.attention(x, x, x)

        # 5. Mean pooling over the 2 tokens
        pooled = attn_output.mean(dim=1)  # [batch, 128]

        # 6. Feedforward
        text_vec = self.ffn(pooled)

        return text_vec

##2.2 Entity News Encoder

`EntityEncoder` integrates four entity-level signals for each news article:
1. KG-based title embeddings
2. KG-based abstract embeddings
3. NER-based title embeddings
4. NER-based abstract embeddings

Each vector is projected into a shared 128-dimensional space and treated as a four-token sequence. There are also binary flags indicating whether each signal is present.

A multi-head self-attention layer models interactions among the available KG and NER tokens while masking out missing ones. The contextualized tokens are then pooled over valid positions and passed through a feedforward layer to produce the final 128‑dimensional entity representation.

In [ ]:
# Each branch embeddings stacked to form embedding matrix
kg_title_matrix = np.stack(news_entities["kg_title_emb"].values)
kg_abstract_matrix = np.stack(news_entities["kg_abstract_emb"].values)
ner_title_matrix = np.stack(news_entities["ner_title_emb"].values)
ner_abstract_matrix = np.stack(news_entities["ner_abstract_emb"].values)
flags_matrix = news_entities[
    ["kg_title_flag", "kg_abstract_flag", "ner_title_flag", "ner_abstract_flag"]
].values.astype(np.float32)

kg_title_tensor = torch.tensor(kg_title_matrix, dtype=torch.float32)
kg_abstract_tensor = torch.tensor(kg_abstract_matrix, dtype=torch.float32)
ner_title_tensor = torch.tensor(ner_title_matrix, dtype=torch.float32)
ner_abstract_tensor = torch.tensor(ner_abstract_matrix, dtype=torch.float32)
flags_tensor = torch.tensor(flags_matrix, dtype=torch.float32)

In [ ]:
class EntityEncoder(nn.Module):
    def __init__(
        self,
        kg_input_dim=100,
        ner_input_dim=384,
        model_dim=128,
        num_heads=2,
        dropout=0.3
    ):
        super().__init__()

        self.kg_proj = nn.Linear(kg_input_dim, model_dim)
        self.ner_proj = nn.Linear(ner_input_dim, model_dim)

        self.attention = nn.MultiheadAttention(
            embed_dim=model_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.ffn = nn.Sequential(
            nn.Linear(model_dim, model_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

    def forward(
        self,
        kg_title_emb,
        kg_abstract_emb,
        ner_title_emb,
        ner_abstract_emb,
        kg_title_flag,
        kg_abstract_flag,
        ner_title_flag,
        ner_abstract_flag
    ):
        kg_title_proj = self.kg_proj(kg_title_emb)
        kg_abstract_proj = self.kg_proj(kg_abstract_emb)
        ner_title_proj = self.ner_proj(ner_title_emb)
        ner_abstract_proj = self.ner_proj(ner_abstract_emb)

        x = torch.stack([
            kg_title_proj,
            kg_abstract_proj,
            ner_title_proj,
            ner_abstract_proj
        ], dim=1)   # [B, 4, 128]

        flags = torch.stack([
            kg_title_flag,
            kg_abstract_flag,
            ner_title_flag,
            ner_abstract_flag
        ], dim=1)

        key_padding_mask = (flags == 0)   # [B, 4]
        all_masked = key_padding_mask.all(dim=1)  # [B]

        # avoid passing fully-masked rows into attention
        safe_key_padding_mask = key_padding_mask.clone()
        safe_key_padding_mask[all_masked] = False

        attn_output, _ = self.attention(
            x, x, x,
            key_padding_mask=safe_key_padding_mask
        )

        mask = (~key_padding_mask).unsqueeze(-1).float()   # [B, 4, 1]
        summed = (attn_output * mask).sum(dim=1)
        count = mask.sum(dim=1).clamp(min=1.0)

        pooled = summed / count

        # explicitly zero out rows where all flags were missing
        pooled = pooled.clone()
        pooled[all_masked] = 0.0

        entity_vec = self.ffn(pooled)

        return entity_vec

In [ ]:
text_input_dim = 768 # BERT embedding dimension
entity_input_dim = 100 # Entity embedding dimension (e.g., from KG)
# num_subclusters is already available from previous execution: num_subclusters = 95

##2.3 Fusion Layer

The `NewsEncoder` constructs the final 128‑dimensional news representation by combining the outputs of the text and entity encoders. It first obtains a semantic vector from the `TextNewsEncoder` and an entity‑level vector from the `EntityEncoder`, then concatenates these two 128‑dimensional representations into a 256‑dimensional vector. A fusion layer projects this fused vector back into a unified 128‑dimensional space, producing the final news representation used by downstream components of the model.

In [ ]:
class NewsEncoder(nn.Module):
    def __init__(
        self,
        text_input_dim=768,  # BERT embeddings are 768-dim
        kg_input_dim=100,
        ner_input_dim=384,
        num_subclusters=96, # Determined from news_features
        model_dim=128,
        num_heads=2,
        dropout=0.3
    ):
        super().__init__()

        self.text_encoder = TextNewsEncoder(
            input_dim=text_input_dim,
            num_subclusters=num_subclusters,
            model_dim=model_dim,
            num_heads=num_heads,
            dropout=dropout
        )

        self.entity_encoder = EntityEncoder(
            kg_input_dim=kg_title_tensor.shape[1],     # 100
            ner_input_dim=ner_title_tensor.shape[1],   # 384
            model_dim=model_dim,
            num_heads=num_heads,
            dropout=dropout
        )

        # Fusion layer to combine text_vec and entity_vec
        self.fusion_layer = nn.Sequential(
            nn.Linear(model_dim * 2, model_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

    def forward(
        self,
        title_abs_emb,
        subcluster_ids,
        kg_title_emb,
        kg_abstract_emb,
        ner_title_emb,
        ner_abstract_emb,
        kg_title_flag,
        kg_abstract_flag,
        ner_title_flag,
        ner_abstract_flag
    ):
        # Encode text features
        text_vec = self.text_encoder(title_abs_emb, subcluster_ids)

        # Encode entity features
        entity_vec = self.entity_encoder(
            kg_title_emb,
            kg_abstract_emb,
            ner_title_emb,
            ner_abstract_emb,
            kg_title_flag,
            kg_abstract_flag,
            ner_title_flag,
            ner_abstract_flag
        )

        # Fuse the text and entity representations
        final_news_vec = self.fusion_layer(torch.cat([text_vec, entity_vec], dim=-1))

        return final_news_vec

#3. User Encoder

### 3.1 Multihead Self-Attention Block for User Representation Learning

The `SelfAttentionBlock` refines the sequence of clicked‑news embeddings by modelling dependencies across the user’s reading history. Rather than treating each news vector independently, it applies a Transformer‑style mechanism that captures interaction patterns and contextual relationships within the sequence.

It performs three key operations:
- **Multi‑head self‑attention**, which computes attention‑weighted interactions between all pairs of news vectors, allowing each item in the history to incorporate information from the others
- **Residual connections and layer normalization**, which stabilise the transformations and preserve the original signal while integrating contextual information
- **A position‑wise feed‑forward network**, which applies a non‑linear projection to each contextualised vector by expanding the hidden dimension fourfold and projecting it back to the original size

Together, these operations transform the raw sequence of clicked‑news embeddings into a context‑aware user representation, where each vector encodes both its own content and its relational structure within the user’s reading history.

In [ ]:
class SelfAttentionBlock(nn.Module):
    def __init__(self, hidden_dim=128, num_head=2, dropout=0.3):
        super().__init__()

        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_head,
            dropout=dropout,
            batch_first=True
        )

        self.dropout1 = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(hidden_dim)

        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.ReLU(),
            nn.Linear(hidden_dim * 4, hidden_dim),
        )
        self.dropout2 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(hidden_dim)

    def forward(self, x, key_padding_mask=None, return_attention_weights=False):
        if key_padding_mask is not None:
            all_padded = key_padding_mask.all(dim=1)   # [B]
            safe_mask = key_padding_mask.clone()
            safe_mask[all_padded] = False
        else:
            all_padded = None
            safe_mask = None

        attn_out, attn_weights = self.attn(
            x, x, x,
            key_padding_mask=safe_mask
        )

        x = self.norm1(x + self.dropout1(attn_out))
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_out))

        if all_padded is not None and all_padded.any():
            x = x.clone()
            x[all_padded] = 0.0

        if return_attention_weights:
            return x, attn_weights
        return x

### 3.2 Additive Attention Layer for User

After the Self‑Attention Block contextualises the clicked‑news sequence, the Additive Attention Layer aggregates these vectors into a single user representation by learning a relevance weight for each item and computing a weighted sum over the sequence.

In [ ]:
class AdditiveAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.proj = nn.Linear(hidden_dim, hidden_dim)
        self.query = nn.Parameter(torch.randn(hidden_dim))

    def forward(self, x, mask=None):
        scores = torch.tanh(self.proj(x)) @ self.query  # (batch, seq_len)

        if mask is not None:
            all_padded = mask.all(dim=1)   # [B]
            safe_mask = mask.clone()
            safe_mask[all_padded] = False

            scores = scores.masked_fill(safe_mask, -1e9)
            weights = torch.softmax(scores, dim=1)

            # zero out weights for rows with no valid history
            weights = weights.masked_fill(mask, 0.0)
            weights = weights / weights.sum(dim=1, keepdim=True).clamp(min=1e-8)
        else:
            all_padded = None
            weights = torch.softmax(scores, dim=1)

        out = (weights.unsqueeze(-1) * x).sum(dim=1)

        if all_padded is not None and all_padded.any():
            out = out.clone()
            out[all_padded] = 0.0

        return out

### 3.3 `UserEncoder`

The `UserEncoder` transforms a user's clicked-news history into a single fixed-sze representation.

Each news item in the history is first encoded using `NewsEncoder`, producing a sequence of 128-dimensional embeddings.

This sequence is then refined by a Self‑Attention Block, which models dependencies and interaction patterns across the user’s reading history.

Finally, an Additive Attention Layer aggregates the contextualised sequence into a single user vector by learning relevance weights over the clicked items. For users with no valid history, the encoder falls back to a learnable default embedding to ensure stable behaviour.


In [ ]:
class UserEncoder(nn.Module):
    def __init__(self, news_encoder, hidden_dim=128, num_head=2, dropout=0.3):
        super().__init__()
        self.news_encoder = news_encoder
        self.attention_block = SelfAttentionBlock(hidden_dim, num_head, dropout)
        self.additive_attention = AdditiveAttention(hidden_dim)

        # Learnable fallback for users with no news reading history
        self.default_user_embedding = nn.Parameter(torch.zeros(hidden_dim))

    def forward(
        self,
        title_abs_emb_hist,
        subcluster_ids_hist,
        kg_title_emb_hist,
        kg_abstract_emb_hist,
        ner_title_emb_hist,
        ner_abstract_emb_hist,
        kg_title_flag_hist,
        kg_abstract_flag_hist,
        ner_title_flag_hist,
        ner_abstract_flag_hist,
        mask=None,
        return_attention_weights=False
    ):

        batch_size, seq_len, _ = title_abs_emb_hist.shape # Assuming these are batched

        # Encode each news item in the history using the comprehensive NewsEncoder
        # Flatten the history dimension to pass all items to NewsEncoder at once
        news_embeddings_flat = self.news_encoder(
            title_abs_emb=title_abs_emb_hist.view(-1, title_abs_emb_hist.size(-1)),
            subcluster_ids=subcluster_ids_hist.view(-1),
            kg_title_emb=kg_title_emb_hist.view(-1, kg_title_emb_hist.size(-1)),
            kg_abstract_emb=kg_abstract_emb_hist.view(-1, kg_abstract_emb_hist.size(-1)),
            ner_title_emb=ner_title_emb_hist.view(-1, ner_title_emb_hist.size(-1)),
            ner_abstract_emb=ner_abstract_emb_hist.view(-1, ner_abstract_emb_hist.size(-1)),
            kg_title_flag=kg_title_flag_hist.view(-1),
            kg_abstract_flag=kg_abstract_flag_hist.view(-1),
            ner_title_flag=ner_title_flag_hist.view(-1),
            ner_abstract_flag=ner_abstract_flag_hist.view(-1)
        )
        news_embeddings = news_embeddings_flat.view(batch_size, seq_len, -1)

        # Self-attention
        if return_attention_weights:
            attended, attn_weights = self.attention_block(
                news_embeddings, key_padding_mask=mask, return_attention_weights=True
            )
        else:
            attended = self.attention_block(
                news_embeddings, key_padding_mask=mask
            )

        # Additive attention pooling
        user_representation = self.additive_attention(attended, mask)

        # Handle users with no history (all padded) on a per-user basis
        if mask is not None:
            all_padded = mask.all(dim=1) # Identify users whose entire history is padded
            if all_padded.any():
                # Replace their representation with the default user embedding
                user_representation = user_representation.clone()
                user_representation[all_padded] = self.default_user_embedding.expand(all_padded.sum(), -1)

        if return_attention_weights:
            return user_representation, attn_weights
        return user_representation

# 4. MLP Input Preparation

Alongside news and user representations, there are several additional feature groups that will be passed into the MLP for our final predictions

These include:
- Additional User Features
- Additonal News Features
- Context Features

##4.1 Additional User Features

The Additional User Features aim to provide more in depth behavioural and preference-level signals that complement the learned user representation.

They include:
*   **Total Historical Clicks (Normalised)** : Normalised Log-Transformed Length of a user’s history
*   **Recent Favourite Category**: Most frequent news category extracted from the user's 10 most recent historical articles.
*   **Top Category 1**:  Top 1 favourite categories for each user from their historical clicks

*   **Top Category 2**: Top 2 favourite categories for each user from their historical clicks
*   **Top Category 3**: Top 3 favourite categories for each user from their historical clicks
*   **Diversity Cluster**: Label to distinguish distinct behaviour types (Cold Start, Low Diversity, Medium Diversity and High Diversity)
*   **Category Cluster** : Places users into clusters based on their top 3 categories.



In [ ]:
# Standardise column names: train uses 'pos_id', val/test use 'pos_ids'
# Make everything use 'pos_id' (scalar) and 'neg_ids' (list of 4)
if 'pos_ids' in train_user_features.columns:
    train_user_features['pos_id'] = train_user_features['pos_ids'].apply(
        lambda x: x[0] if isinstance(x, (list, np.ndarray)) else x
    )
if 'pos_ids' in val_user_features.columns:
    val_user_features['pos_id'] = val_user_features['pos_ids'].apply(
        lambda x: x[0] if isinstance(x, (list, np.ndarray)) else x
    )
if 'pos_ids' in test_user_features.columns:
    test_user_features['pos_id'] = test_user_features['pos_ids'].apply(
        lambda x: x[0] if isinstance(x, (list, np.ndarray)) else x
    )

In [ ]:
numerical_user_features = [
    'history_length',
    'diversity_ratio',
    'concentration_ratio'
]

categorical_user_features = [
    'recent_fav_cat',
    'top_cat_1',
    'top_cat_2',
    'top_cat_3',
    'diversity_cluster', # Moved here
    'interest_cluster'   # Moved here
]

# Filter features that actually exist in the DataFrame
selected_numerical_user_features = [f for f in numerical_user_features if f in train_user_features.columns]
selected_categorical_user_features = [f for f in categorical_user_features if f in train_user_features.columns]

print(f"Numerical user features: {selected_numerical_user_features}")
print(f"Categorical user features: {selected_categorical_user_features}")

Numerical user features: ['history_length', 'diversity_ratio', 'concentration_ratio']
Categorical user features: ['recent_fav_cat', 'top_cat_1', 'top_cat_2', 'top_cat_3', 'diversity_cluster', 'interest_cluster']


###4.1.1 Preprocessing User Features

In [ ]:
def preprocess_user_features(df, numerical_cols, categorical_cols, categorical_mappers=None, scaler=None):

    # 1. Extract numerical features
    user_numerical_data = df[numerical_cols].values.astype(np.float32)

    # 2. Log Transform history_length to fix the heavy-tail skew
    if 'history_length' in numerical_cols:
        hl_idx = numerical_cols.index('history_length')
        user_numerical_data[:, hl_idx] = np.log1p(user_numerical_data[:, hl_idx])

    # 3. Apply Standard Scaler
    if scaler is None:
        scaler = StandardScaler()
        user_numerical_data = scaler.fit_transform(user_numerical_data).astype(np.float32)
    else:
        user_numerical_data = scaler.transform(user_numerical_data).astype(np.float32)

    # 4. Categorical Encoding
    user_categorical_encoded_df = pd.DataFrame(index=df.index)
    current_categorical_mappers = {}
    vocab_sizes = {}

    if categorical_mappers is None: # Learn mappers from training data
        for col in categorical_cols:
            codes, uniques = pd.factorize(df[col].astype(str))
            user_categorical_encoded_df[col] = codes
            current_categorical_mappers[col] = {category: i for i, category in enumerate(uniques)}
            vocab_sizes[col] = len(uniques)
        final_mappers_to_return = current_categorical_mappers
    else: # Apply existing mappers to validation/test data
        final_mappers_to_return = categorical_mappers
        for col in categorical_cols:
            mapper = categorical_mappers[col]
            mapped_codes = df[col].astype(str).map(mapper).fillna(len(mapper)).astype(int)
            user_categorical_encoded_df[col] = mapped_codes
            vocab_sizes[col] = len(mapper) + 1

    # Return the scaler as the 5th variable
    return user_numerical_data, user_categorical_encoded_df, final_mappers_to_return, vocab_sizes, scaler

**Apply preprocessing to training data**

In [ ]:
# Process training data and capture the categorical mappers, vocab sizes, AND scaler
train_user_numerical_input, train_user_categorical_input, user_category_mappers, user_vocab_sizes, trained_scaler = preprocess_user_features(
    train_user_features,
    selected_numerical_user_features,
    selected_categorical_user_features
)

user_numerical_dense_dim = train_user_numerical_input.shape[1]

print(f"Shape of train_user_numerical_input: {train_user_numerical_input.shape}")
print(f"Shape of train_user_categorical_input: {train_user_categorical_input.shape}")
print(f"User numerical dense dimension: {user_numerical_dense_dim}")
print(f"Learned categorical mappers (first 5 for each category):")
for col, mapper in user_category_mappers.items():
    print(f"  {col}: {list(mapper.items())[:5]} ... (total {len(mapper)})")
print(f"Learned vocabulary sizes: {user_vocab_sizes}")

Shape of train_user_numerical_input: (141558, 3)
Shape of train_user_categorical_input: (141558, 6)
User numerical dense dimension: 3
Learned categorical mappers (first 5 for each category):
  recent_fav_cat: [('tvnews', 0), ('travelnews', 1), ('tv-celebrity', 2), ('autos_general', 3), ('newsgoodnews', 4)] ... (total 96)
  top_cat_1: [('tvnews', 0), ('ViolentCrime', 1), ('tv-celebrity', 2), ('autos_general', 3), ('US_ColdCases_PoliceMisconduct', 4)] ... (total 96)
  top_cat_2: [('2020Election_LegislativeReform', 0), ('PolicePursuits_DrugCrime', 1), ('markets', 2), ('travelnews', 3), ('CounterTerrorism_MiddleEastConflict', 4)] ... (total 96)
  top_cat_3: [('MLB_Awards_Rankings', 0), ('US_ColdCases_PoliceMisconduct', 1), ('basketball_nba', 2), ('ClimateInfrastructure_Pollution', 3), ('newsgoodnews', 4)] ... (total 96)
  diversity_cluster: [('0', 0), ('1', 1), ('2', 2), ('-1', 3)] ... (total 4)
  interest_cluster: [('0', 0), ('2', 1), ('1', 2), ('3', 3)] ... (total 4)
Learned vocabulary s

**Apply preprocessing to validation and test data**

In [ ]:
# Pass the trained_scaler in to properly normalize validation and test data
val_user_numerical_input, val_user_categorical_input, _, _, _ = preprocess_user_features(
    val_user_features,
    selected_numerical_user_features,
    selected_categorical_user_features,
    categorical_mappers=user_category_mappers,
    scaler=trained_scaler # Added scaler
)

test_user_numerical_input, test_user_categorical_input, _, _, _ = preprocess_user_features(
    test_user_features,
    selected_numerical_user_features,
    selected_categorical_user_features,
    categorical_mappers=user_category_mappers,
    scaler=trained_scaler # Added scaler
)

print(f"Shape of val_user_numerical_input: {val_user_numerical_input.shape}")
print(f"Shape of val_user_categorical_input: {val_user_categorical_input.shape}")
print(f"Shape of test_user_numerical_input: {test_user_numerical_input.shape}")
print(f"Shape of test_user_categorical_input: {test_user_categorical_input.shape}")

Shape of val_user_numerical_input: (31624, 3)
Shape of val_user_categorical_input: (31624, 6)
Shape of test_user_numerical_input: (30270, 3)
Shape of test_user_categorical_input: (30270, 6)


In [ ]:
# Convert user numerical features to tensors
train_user_numerical_tensor = torch.from_numpy(train_user_numerical_input)
val_user_numerical_tensor = torch.from_numpy(val_user_numerical_input)
test_user_numerical_tensor = torch.from_numpy(test_user_numerical_input)

# Convert user categorical features (DataFrame of integer codes) to tensors
train_user_categorical_tensors = {col: torch.from_numpy(train_user_categorical_input[col].values) for col in selected_categorical_user_features}
val_user_categorical_tensors = {col: torch.from_numpy(val_user_categorical_input[col].values) for col in selected_categorical_user_features}
test_user_categorical_tensors = {col: torch.from_numpy(test_user_categorical_input[col].values) for col in selected_categorical_user_features}

print(f"Train user numerical tensor shape: {train_user_numerical_tensor.shape}")
print(f"Train user categorical tensors (example): {train_user_categorical_tensors[selected_categorical_user_features[0]].shape}")

Train user numerical tensor shape: torch.Size([141558, 3])
Train user categorical tensors (example): torch.Size([141558])


##4.2 Additional News Features

*   **Latent Topics:** Using SVD-reduced TF-IDF vectors, we derived the top 50 most important latent topics. This feature shows the score for each article for each topic. They aim to help the model understand what is the article about beyond the leaned news embedding, enhancing its ability to match articles to users.

In [ ]:
latent_topics = news_features.filter(regex='topic_').to_numpy(dtype=np.float32)
topics_tensor = torch.from_numpy(latent_topics)
# Global lookup: news_id → integer index into the news feature tensors
news_id_to_idx_map = {nid: idx for idx, nid in enumerate(news_features['news_id'])}

##4.3 Context Features

*  **Hour of the Day (Sine)**: Cyclical encoding of the hour at which the impression occurred, using sine transformations to avoid discontinuities at midnight.

*  **Hour of the Day (Cosine)**:
Cyclical encoding of the hour at which the impression occurred, using cosine transformations to avoid discontinuities at midnight.


*  **Recency of News Article**: A freshness signal that gives higher values to newer articles and lower values to older ones. It is computed as a list of scores aligned to the candidate articles, where each score is calculated as 1 / (1 + time since the user’s first impression).

*  **Habit Score**: Captures whether the user is returning at their habitual time of day. It is computed as the cosine transformation of the hour interval between the current impression and the user’s previous impression, allowing the model to recognise consistent daily reading patterns.

In [ ]:
train_context_scalar = torch.tensor(
    train_context_features[['hour_sin', 'hour_cos', 'habit_score']].values.astype(np.float32)
)  # [N_train, 3]

val_context_scalar = torch.tensor(
    val_context_features[['hour_sin', 'hour_cos', 'habit_score']].values.astype(np.float32)
)  # [N_val, 3]

test_context_scalar = torch.tensor(
    test_context_features[['hour_sin', 'hour_cos', 'habit_score']].values.astype(np.float32)
)  # [N_test, 3]

# ── Recency score: train has fixed 5 candidates, stack into tensor ────────────
train_recency_score = torch.tensor(
    np.stack(train_context_features['recency_scores'].values),
    dtype=torch.float32
)  # [N_train, 5]

# ── Recency score: val/test have variable candidates, use lookup ──────────────
def build_recency_lookup(context_df):
    """Returns {(impression_id, news_id): recency_scores}"""
    lookup = {}
    for _, row in context_df.iterrows():
        imp_id     = row['impression_id']
        candidates = row['candidate_articles'] # Corrected column name
        scores     = row['recency_scores']
        for news_id, score in zip(candidates, scores):
            lookup[(imp_id, news_id)] = score
    return lookup

val_recency_lookup  = build_recency_lookup(val_context_features)
test_recency_lookup = build_recency_lookup(test_context_features)

print(f"train_context_scalar shape:  {train_context_scalar.shape}")
print(f"train_recency_score shape:   {train_recency_score.shape}")
print(f"val recency lookup size:     {len(val_recency_lookup)}")

train_context_scalar shape:  torch.Size([141558, 3])
train_recency_score shape:   torch.Size([141558, 5])
val recency lookup size:     1215336


# 5. MLP Implementation

The MLP implementation is similar to that of previous iterations

In [ ]:
class NewsMLP(nn.Module):
    """
    Scores a single (user, news) candidate pair -> scalar relevance score.
    """
    def __init__(
        self,
        user_numerical_dense_dim,
        user_categorical_vocab_sizes,
        topic_dim=50,
        context_dim=4,   # [hour_sin, hour_cos, habit_score, recency]
        user_repr_dim=128,
        news_repr_dim=128,
        hidden_dims=[128, 64, 32],
        embed_dim=128,
    ):
        super().__init__()

        self.user_repr_proj = nn.Linear(user_repr_dim, embed_dim)
        self.news_repr_proj = nn.Linear(news_repr_dim, embed_dim)

        self.user_numerical_dense_proj = nn.Linear(user_numerical_dense_dim, embed_dim)

        self.user_categorical_embeddings = nn.ModuleDict({
            col: nn.Embedding(vocab_size + 1, embed_dim)
            for col, vocab_size in user_categorical_vocab_sizes.items()
        })

        self.topic_proj = nn.Linear(topic_dim, embed_dim)

        self.context_proj = nn.Linear(context_dim, embed_dim)

        num_user_cat = len(user_categorical_vocab_sizes)

        total_input_dim = embed_dim * (
            1 +  # user_repr
            1 +  # news_repr
            1 +  # user_numerical
            num_user_cat +
            1 +  # topic
            1    # context_features
        )

        self.fc1 = nn.Linear(total_input_dim, hidden_dims[0])
        self.fc2 = nn.Linear(hidden_dims[0], hidden_dims[1])
        self.fc3 = nn.Linear(hidden_dims[1], hidden_dims[2])
        self.output = nn.Linear(hidden_dims[2], 1)

        self.relu = nn.ReLU()

    def forward(
        self,
        user_repr,
        news_repr,
        user_numerical_dense,
        user_categorical_input,
        topic_embed,
        context_features,   # [B, 4]
    ):
        user_repr_vec = self.relu(self.user_repr_proj(user_repr))
        news_repr_vec = self.relu(self.news_repr_proj(news_repr))

        user_numerical_vec = self.relu(self.user_numerical_dense_proj(user_numerical_dense))

        user_categorical_vecs = [
            self.user_categorical_embeddings[col](user_categorical_input[col].long())
            for col in self.user_categorical_embeddings.keys()
        ]

        topic_vec = self.relu(self.topic_proj(topic_embed))

        context_vec = self.relu(self.context_proj(context_features))

        x = torch.cat([
            user_repr_vec,
            news_repr_vec,
            user_numerical_vec,
            *user_categorical_vecs,
            topic_vec,
            context_vec,
        ], dim=1)

        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))

        score = self.output(x)
        return score.squeeze(1)

#6. Preparing Data

## 6.1 Initialise Classes to Prepare Datasets

There are 2 classes we've created to prepare our training, validation and testing datasets.

1. `HybridNewsDataset` class is used to prepare our training sets, constructing one training sample per impression by assembling all the inputs required by the model. For each impression, it gathers the users' numerical and categorical features, builds 5 candicate news items **(1 positive + 4 negatives news items)** with their raw encoder inputs. The candidate articles are then randomly shuffled to ensure that the positive item does not always appear in a fixed position, and the corresponding label is updated accordingly. It also prepares user's click history and attaches the other additional features such as latent topics and context signals (hour, habit score, recenecy). Finally, it packages all the components in a structured dictionary that can be directly consumed by the mdeol during training.

2. `HybridNewsDatasetVariable` class prepares the validation and testing samples, where each impression has a **variable number of candidate news articles**. For every impression, it retrieves the user’s numerical and categorical features, extracts raw encoder inputs for all candidate articles, and prepares the user’s padded click history. It also attaches additional candidate‑level features such as latent topic vectors, context signals (hour, habit score), and recency scores computed per (impression, news) pair. Finally, it returns all components in a structured dictionary, along with the ground‑truth labels for each candidate, enabling the model to evaluate ranking performance under realistic, variable‑sized candidate sets.

In [ ]:
class HybridNewsDataset(Dataset):
    """
    One item = one impression group.

    Returns:
      - candidate_inputs: raw encoder inputs for 5 candidates
      - history_inputs: raw encoder inputs for padded history
      - user_numerical
      - user_categorical
      - topic
      - context_features
      - label (always 0 since positive is first)
    """

    def __init__(
        self,
        user_numerical_features,         # tensor [N, user_num_dim]
        user_categorical_features_dict,  # dict {col: tensor [N]}
        pos_news_ids,                    # pd.Series [N]
        neg_news_ids,                    # pd.Series [N], each row list of 4
        history_news_ids,                # pd.Series/list [N], each row list of clicked history ids
        impression_ids,                  # pd.Series [N]
        news_id_to_idx_map,              # dict {news_id: idx}

        # Raw news encoder inputs
        all_title_abs_embeddings,        # tensor [M, text_dim]
        all_subcluster_ids,              # tensor [M]
        all_kg_title_embeddings,         # tensor [M, ent_dim]
        all_kg_abstract_embeddings,      # tensor [M, ent_dim]
        all_ner_title_embeddings,        # tensor [M, ent_dim]
        all_ner_abstract_embeddings,     # tensor [M, ent_dim]
        all_flags_matrix,                # tensor [M, 4] ordered:
                                         # [kg_title, kg_abstract, ner_title, ner_abstract]

        # Extra features for final scorer
        all_topic_embeddings,            # tensor [M, topic_dim]
        context_scalar,                  # tensor [N, 3] (hour_sin, hour_cos, habit_score)
        recency_scores,                  # tensor [N, 5]

        max_history_len=10,
        pad_news_id=None,                # optional explicit pad id
    ):
        self.user_numerical = user_numerical_features
        self.user_categorical = user_categorical_features_dict

        self.pos_news_ids = pos_news_ids.reset_index(drop=True)
        self.neg_news_ids = neg_news_ids.reset_index(drop=True)
        self.history_news_ids = history_news_ids.reset_index(drop=True)
        self.impression_ids = impression_ids.reset_index(drop=True)

        self.news_id_to_idx = news_id_to_idx_map

        # Raw encoder inputs
        self.title_abs = all_title_abs_embeddings
        self.subcluster = all_subcluster_ids
        self.kg_title = all_kg_title_embeddings
        self.kg_abs = all_kg_abstract_embeddings
        self.ner_title = all_ner_title_embeddings
        self.ner_abs = all_ner_abstract_embeddings
        self.flags = all_flags_matrix

        # Extra scorer features
        self.topic = all_topic_embeddings
        self.context_scalar = context_scalar
        self.recency_scores = recency_scores

        self.max_history_len = max_history_len

        if pad_news_id is not None:
            self.pad_news_id = pad_news_id
        else:
            # fallback: first news id in map
            self.pad_news_id = next(iter(news_id_to_idx_map.keys()))

    def __len__(self):
        return len(self.pos_news_ids)

    def _normalize_news_id(self, news_id):
        if isinstance(news_id, str):
            if news_id.startswith("N"):
                return int(news_id[1:])
            if news_id.isdigit():
                return int(news_id)
        return news_id

    def _get_news_index(self, news_id):
        news_id = self._normalize_news_id(news_id)
        idx = self.news_id_to_idx.get(news_id, None)
        if idx is None:
            raise ValueError(f"News ID {news_id} not found in news_id_to_idx_map.")
        return idx

    def _get_raw_news_features(self, news_id):
        idx = self._get_news_index(news_id)

        return {
            "title_abs_emb": self.title_abs[idx],          # [text_dim]
            "subcluster_ids": self.subcluster[idx],        # scalar
            "kg_title_emb": self.kg_title[idx],            # [ent_dim]
            "kg_abstract_emb": self.kg_abs[idx],           # [ent_dim]
            "ner_title_emb": self.ner_title[idx],          # [ent_dim]
            "ner_abstract_emb": self.ner_abs[idx],         # [ent_dim]
            "kg_title_flag": self.flags[idx][0],           # scalar
            "kg_abstract_flag": self.flags[idx][1],        # scalar
            "ner_title_flag": self.flags[idx][2],          # scalar
            "ner_abstract_flag": self.flags[idx][3],       # scalar
        }

    def _build_padded_history(self, history_ids):
        """
        Returns:
          padded_history_ids: length H
          mask: length H, True = padded, False = valid
        """
        cleaned = []
        for nid in history_ids:
            nid = self._normalize_news_id(nid)
            if nid in self.news_id_to_idx:
                cleaned.append(nid)

        cleaned = cleaned[-self.max_history_len:]

        pad_len = self.max_history_len - len(cleaned)
        padded_history_ids = [self.pad_news_id] * pad_len + cleaned
        mask = [True] * pad_len + [False] * len(cleaned)

        return padded_history_ids, mask

    def __getitem__(self, idx):
        # ---------- user ----------
        user_num = self.user_numerical[idx]  # [user_num_dim]
        user_cat = {
            col: self.user_categorical[col][idx]
            for col in self.user_categorical
        }

        # ---------- candidate ids ----------
        pos_id = self.pos_news_ids.iloc[idx]
        neg_ids = self.neg_news_ids.iloc[idx]
        candidate_ids = [pos_id] + list(neg_ids)   # [5]
        perm = np.random.permutation(len(candidate_ids))
        candidate_ids = [candidate_ids[i] for i in perm]

        # ---------- candidate raw encoder inputs ----------
        candidate_feature_dicts = [
            self._get_raw_news_features(nid)
            for nid in candidate_ids
        ]

        candidate_inputs = {
            key: torch.stack([feat[key] for feat in candidate_feature_dicts])
            for key in candidate_feature_dicts[0]
        }
        # shapes:
        # title_abs_emb      [5, text_dim]
        # subcluster_ids     [5]
        # kg_title_emb       [5, ent_dim]
        # ...
        # flags              [5]

        # ---------- history raw encoder inputs ----------
        history_ids = self.history_news_ids.iloc[idx]
        padded_history_ids, hist_mask = self._build_padded_history(history_ids)

        history_feature_dicts = [
            self._get_raw_news_features(nid)
            for nid in padded_history_ids
        ]

        history_inputs = {
            key + "_hist": torch.stack([feat[key] for feat in history_feature_dicts])
            for key in history_feature_dicts[0]
        }
        history_inputs["mask"] = torch.tensor(hist_mask, dtype=torch.bool)  # [H]

        # ---------- candidate-level extra features ----------
        candidate_indices = [self._get_news_index(nid) for nid in candidate_ids]

        topic = self.topic[candidate_indices]             # [5, topic_dim]

        recency = self.recency_scores[idx].unsqueeze(1)       # [5, 1]
        ctx_scalar = self.context_scalar[idx]                 # [3]
        ctx_scalar_expanded = ctx_scalar.unsqueeze(0).expand(len(candidate_ids), -1)  # [5, 3]
        context_features = torch.cat([ctx_scalar_expanded, recency], dim=1)           # [5, 4]

        # positive always first
        label = torch.tensor(np.where(perm == 0)[0][0], dtype=torch.long)

        return {
            "candidate_inputs": candidate_inputs,
            "history_inputs": history_inputs,
            "user_numerical": user_num,
            "user_categorical": user_cat,
            "topic": topic,
            "context_features": context_features,
            "label": label,
            "impression_id": self.impression_ids.iloc[idx],
        }

class HybridNewsDatasetVariable(Dataset):
    """
    Val/test dataset for variable number of candidates per impression.

    Returns:
      - candidate_inputs: raw encoder inputs for all candidates [C, ...]
      - history_inputs: raw encoder inputs for padded history [H, ...]
      - user_numerical
      - user_categorical
      - topic
      - context_features     [C, 4] = [hour_sin, hour_cos, habit_score, recency]
      - labels               [C]
      - impression_id
    """

    def __init__(
        self,
        user_numerical_features,         # tensor [N, user_num_dim]
        user_categorical_features_dict,  # dict {col: tensor [N]}
        interactions_df,                 # val_interactions / test_interactions
        history_news_ids,                # pd.Series/list [N], each row list of clicked history ids
        impression_ids,                  # pd.Series [N], aligned with user/context tables
        news_id_to_idx_map,

        # Raw news encoder inputs
        all_title_abs_embeddings,        # tensor [M, text_dim]
        all_subcluster_ids,              # tensor [M]
        all_kg_title_embeddings,         # tensor [M, ent_dim]
        all_kg_abstract_embeddings,      # tensor [M, ent_dim]
        all_ner_title_embeddings,        # tensor [M, ent_dim]
        all_ner_abstract_embeddings,     # tensor [M, ent_dim]
        all_flags_matrix,                # tensor [M, 4]
                                         # [kg_title, kg_abstract, ner_title, ner_abstract]

        # Extra scorer features
        all_topic_embeddings,            # tensor [M, topic_dim]
        context_scalar_tensor,           # tensor [N, 3]
        recency_lookup,                  # dict {(impression_id, news_id): score}

        max_history_len=10,
        pad_news_id=None,
    ):
        self.user_numerical = user_numerical_features
        self.user_categorical = user_categorical_features_dict
        self.interactions = interactions_df.reset_index(drop=True)
        self.history_news_ids = history_news_ids.reset_index(drop=True)
        self.impression_ids = impression_ids.reset_index(drop=True)

        self.news_id_to_idx = news_id_to_idx_map

        self.title_abs = all_title_abs_embeddings
        self.subcluster = all_subcluster_ids
        self.kg_title = all_kg_title_embeddings
        self.kg_abs = all_kg_abstract_embeddings
        self.ner_title = all_ner_title_embeddings
        self.ner_abs = all_ner_abstract_embeddings
        self.flags = all_flags_matrix

        self.topic = all_topic_embeddings

        self.context_scalar = context_scalar_tensor
        self.recency_lookup = recency_lookup

        self.max_history_len = max_history_len
        self.pad_news_id = pad_news_id if pad_news_id is not None else next(iter(news_id_to_idx_map.keys()))

    def __len__(self):
        return len(self.interactions)

    def _normalize_news_id(self, news_id):
        if isinstance(news_id, str):
            if news_id.startswith("N"):
                return int(news_id[1:])
            if news_id.isdigit():
                return int(news_id)
        return news_id

    def _get_news_index(self, news_id):
        news_id = self._normalize_news_id(news_id)
        idx = self.news_id_to_idx.get(news_id, None)
        if idx is None:
            raise ValueError(f"News ID {news_id} not found in news_id_to_idx_map.")
        return idx

    def _get_raw_news_features(self, news_id):
        idx = self._get_news_index(news_id)
        return {
            "title_abs_emb": self.title_abs[idx],
            "subcluster_ids": self.subcluster[idx],
            "kg_title_emb": self.kg_title[idx],
            "kg_abstract_emb": self.kg_abs[idx],
            "ner_title_emb": self.ner_title[idx],
            "ner_abstract_emb": self.ner_abs[idx],
            "kg_title_flag": self.flags[idx][0],
            "kg_abstract_flag": self.flags[idx][1],
            "ner_title_flag": self.flags[idx][2],
            "ner_abstract_flag": self.flags[idx][3],
        }

    def _build_padded_history(self, history_ids):
        cleaned = []
        for nid in history_ids:
            nid = self._normalize_news_id(nid)
            if nid in self.news_id_to_idx:
                cleaned.append(nid)

        cleaned = cleaned[-self.max_history_len:]
        pad_len = self.max_history_len - len(cleaned)

        padded_history_ids = [self.pad_news_id] * pad_len + cleaned
        mask = [True] * pad_len + [False] * len(cleaned)   # True = padded

        return padded_history_ids, mask

    def __getitem__(self, idx):
        row = self.interactions.iloc[idx]
        impression_id = row["impression_id"]
        candidate_ids = row["candidate_news"]   # variable length C
        labels = row["labels"]                  # variable length C

        # User features
        user_num = self.user_numerical[idx]
        user_cat = {
            col: self.user_categorical[col][idx]
            for col in self.user_categorical
        }

        # Candidate raw encoder inputs
        candidate_feature_dicts = [
            self._get_raw_news_features(nid)
            for nid in candidate_ids
        ]

        candidate_inputs = {
            key: torch.stack([feat[key] for feat in candidate_feature_dicts])
            for key in candidate_feature_dicts[0]
        }
        # candidate_inputs[key]: [C, ...]

        # History raw encoder inputs
        history_ids = self.history_news_ids.iloc[idx]
        padded_history_ids, hist_mask = self._build_padded_history(history_ids)

        history_feature_dicts = [
            self._get_raw_news_features(nid)
            for nid in padded_history_ids
        ]

        history_inputs = {
            key + "_hist": torch.stack([feat[key] for feat in history_feature_dicts])
            for key in history_feature_dicts[0]
        }
        history_inputs["mask"] = torch.tensor(hist_mask, dtype=torch.bool)  # [H]

        # Extra candidate-level features
        candidate_indices = [self._get_news_index(nid) for nid in candidate_ids]

        topic = self.topic[candidate_indices]             # [C, topic_dim]

        recency = torch.tensor(
            [self.recency_lookup.get((impression_id, nid), 0.0) for nid in candidate_ids],
            dtype=torch.float32
        ).unsqueeze(1)                                    # [C, 1]

        ctx_scalar = self.context_scalar[idx]             # [3]
        ctx_scalar_expanded = ctx_scalar.unsqueeze(0).expand(len(candidate_ids), -1)  # [C, 3]
        context_features = torch.cat([ctx_scalar_expanded, recency], dim=1)            # [C, 4]

        labels_tensor = torch.tensor(labels, dtype=torch.float32)  # [C]

        return {
            "candidate_inputs": candidate_inputs,
            "history_inputs": history_inputs,
            "user_numerical": user_num,
            "user_categorical": user_cat,
            "topic": topic,
            "context_features": context_features,
            "labels": labels_tensor,
            "impression_id": impression_id,
        }


def hybrid_collate_fn(batch):
    candidate_inputs_list = [b["candidate_inputs"] for b in batch]
    history_inputs_list = [b["history_inputs"] for b in batch]

    user_nums = [b["user_numerical"] for b in batch]
    user_cats = [b["user_categorical"] for b in batch]

    topic_list = [b["topic"] for b in batch]
    context_list = [b["context_features"] for b in batch]
    labels = [b["label"] for b in batch]
    impression_ids = [b["impression_id"] for b in batch]

    candidate_inputs_batch = {
        key: torch.stack([x[key] for x in candidate_inputs_list])
        for key in candidate_inputs_list[0]
    }
    # [B, 5, ...]

    history_inputs_batch = {
        key: torch.stack([x[key] for x in history_inputs_list])
        for key in history_inputs_list[0]
    }
    # [B, H, ...]

    user_num_batch = torch.stack(user_nums)  # [B, user_num_dim]

    user_cat_batch = {
        col: torch.stack([uc[col] for uc in user_cats])
        for col in user_cats[0]
    }  # {col: [B]}

    topic_batch = torch.stack(topic_list)              # [B, 5, topic_dim]
    context_batch = torch.stack(context_list)          # [B, 5, 4]
    label_batch = torch.stack(labels)                  # [B]

    return {
        "candidate_inputs": candidate_inputs_batch,
        "history_inputs": history_inputs_batch,
        "user_numerical": user_num_batch,
        "user_categorical": user_cat_batch,
        "topic": topic_batch,
        "context_features": context_batch,
        "label": label_batch,
        "impression_id": impression_ids,
    }

def hybrid_variable_collate_fn(batch):
    item = batch[0]

    return {
        "candidate_inputs": {
            k: v.unsqueeze(0) for k, v in item["candidate_inputs"].items()
        },  # [1, C, ...]
        "history_inputs": {
            k: v.unsqueeze(0) for k, v in item["history_inputs"].items()
        },  # [1, H, ...]
        "user_numerical": item["user_numerical"].unsqueeze(0),  # [1, user_num_dim]
        "user_categorical": {
            col: val.unsqueeze(0) for col, val in item["user_categorical"].items()
        },  # {col: [1]}
        "topic": item["topic"].unsqueeze(0),                    # [1, C, topic_dim]
        "context_features": item["context_features"].unsqueeze(0),  # [1, C, 4]
        "labels": item["labels"].unsqueeze(0),                  # [1, C]
        "impression_id": [item["impression_id"]],
    }

## 6.2 Build Train, Validation and Test Datasets

In [ ]:
# Build datasets
# NOTE: train_interactions must have 'pos_id' (scalar) and 'neg_ids' (list of 4)
#       at the impression level, matching the row order of train_user_features.

train_dataset = HybridNewsDataset(
    user_numerical_features=train_user_numerical_tensor,
    user_categorical_features_dict=train_user_categorical_tensors,
    pos_news_ids=train_user_features["pos_id"],
    neg_news_ids=train_user_features["neg_ids"],
    history_news_ids=train_user_features["history"],
    impression_ids=train_user_features["impression_id"],
    news_id_to_idx_map=news_id_to_idx_map,

    all_title_abs_embeddings=title_abs_embedding_tensor,
    all_subcluster_ids=subcluster_id_tensor,
    all_kg_title_embeddings=kg_title_tensor,
    all_kg_abstract_embeddings=kg_abstract_tensor,
    all_ner_title_embeddings=ner_title_tensor,
    all_ner_abstract_embeddings=ner_abstract_tensor,
    all_flags_matrix=flags_tensor,

    all_topic_embeddings=topics_tensor,

    context_scalar=train_context_scalar,
    recency_scores=train_recency_score,                   # now used

    max_history_len=10,
)

val_dataset = HybridNewsDatasetVariable(
    user_numerical_features=val_user_numerical_tensor,
    user_categorical_features_dict=val_user_categorical_tensors,
    interactions_df=val_interactions,
    history_news_ids=val_user_features["history"],
    impression_ids=val_user_features["impression_id"],
    news_id_to_idx_map=news_id_to_idx_map,

    all_title_abs_embeddings=title_abs_embedding_tensor,
    all_subcluster_ids=subcluster_id_tensor,
    all_kg_title_embeddings=kg_title_tensor,
    all_kg_abstract_embeddings=kg_abstract_tensor,
    all_ner_title_embeddings=ner_title_tensor,
    all_ner_abstract_embeddings=ner_abstract_tensor,
    all_flags_matrix=flags_tensor,

    all_topic_embeddings=topics_tensor,
    context_scalar_tensor=val_context_scalar,
    recency_lookup=val_recency_lookup,

    max_history_len=10,
)

test_dataset = HybridNewsDatasetVariable(
    user_numerical_features=test_user_numerical_tensor,
    user_categorical_features_dict=test_user_categorical_tensors,
    interactions_df=test_interactions,
    history_news_ids=test_user_features["history"],
    impression_ids=test_user_features["impression_id"],
    news_id_to_idx_map=news_id_to_idx_map,

    all_title_abs_embeddings=title_abs_embedding_tensor,
    all_subcluster_ids=subcluster_id_tensor,
    all_kg_title_embeddings=kg_title_tensor,
    all_kg_abstract_embeddings=kg_abstract_tensor,
    all_ner_title_embeddings=ner_title_tensor,
    all_ner_abstract_embeddings=ner_abstract_tensor,
    all_flags_matrix=flags_tensor,

    all_topic_embeddings=topics_tensor,
    context_scalar_tensor=test_context_scalar,
    recency_lookup=test_recency_lookup,

    max_history_len=10,
)

#7. Training NRMS

This section focuses on training the NRMS model performance. To understand how different architectural and regularisation choices impact performance , we train four distinct model configurations.

We vary only the number of attention heads and the dropout rate while keeping all other hyperparameters (e.g. training epochs, MLP variables, learning rate) constant.

For each configuration, we report the full set of ranking metrics (MRR, MAP, AUC, nDCG@5, nDCG@10) on both the training and test splits. This setup allows us to systematically compare how model capacity and regularisation strength influence generalisation across the four trained variants.

The table below summarises the four model configurations we have trained and the training results:

| Test No. | Model Configuration                         | MRR | MAP | AUC | nDCG@5 | nDCG@10 |
|----------|---------------------------------------------|-----|-----|-----|--------|---------|
| 1        | 4 Heads; Dropout 0.3; 10 Epochs             | 0.6638 | 0.6638 | 0.7639 | 0.7476    | 0.7476     |
| 2        | 2 Heads; Dropout 0.1; 10 Epochs             | 0.6655 | 0.6655 | 0.7653 | 0.7489    | 0.7489     |
| 3        | 2 Heads; Dropout 0.2; 10 Epochs             | 0.6580 | 0.6580 | 0.7572 | 0.7432    | 0.7432     |
| 4        | 2 Heads; Dropout 0.3; 10 Epochs             | 0.6547 | 0.6547 | 0.7534 | 0.7407    | 0.7407     |



In [ ]:
class NRMSModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.news_encoder = NewsEncoder(
            text_input_dim=title_abs_embedding_tensor.shape[1],
            kg_input_dim=kg_title_tensor.shape[1],
            ner_input_dim=ner_title_tensor.shape[1],
            num_subclusters=int(subcluster_id_tensor.max().item()) + 1,
            model_dim=128,
            num_heads=2,
            dropout=0.3,
        )
        self.user_encoder = UserEncoder(self.news_encoder)

        self.mlp = NewsMLP(
            user_repr_dim=128,
            news_repr_dim=128,
            user_numerical_dense_dim=user_numerical_dense_dim,
            user_categorical_vocab_sizes=user_vocab_sizes,
            topic_dim=topics_tensor.shape[1],
            context_dim=4,   # [hour_sin, hour_cos, habit_score, recency]
            hidden_dims=[128, 64, 32],
            embed_dim=128
        )

    def forward(
        self,
        candidate_inputs,         # dict, each [B, 5, ...]
        history_inputs,           # dict, each [B, H, ...]
        user_numerical_dense,     # [B, user_num_dim]
        user_categorical_input,   # dict {col: [B]}
        topic_embed,              # [B, 5, topic_dim]
        context_features,         # [B, 5, 4]
    ):
        batch_size = user_numerical_dense.size(0)
        num_candidates = topic_embed.size(1)   # usually 5

        # ─────────────────────────────
        # 1. Encode user from history
        # ─────────────────────────────
        user_repr = self.user_encoder(**history_inputs)   # [B, 128]

        # ─────────────────────────────
        # 2. Encode candidate news
        # flatten [B, 5, ...] -> [B*5, ...]f
        # ─────────────────────────────
        candidate_inputs_flat = {
            key: value.reshape(-1, *value.shape[2:]) if value.dim() > 2 else value.reshape(-1)
            for key, value in candidate_inputs.items()
        }

        news_repr_flat = self.news_encoder(**candidate_inputs_flat)   # [B*5, 128]
        news_repr = news_repr_flat.reshape(batch_size, num_candidates, -1)  # [B, 5, 128]

        # ─────────────────────────────
        # 3. Expand user-side features to candidate level
        # ─────────────────────────────
        user_repr_expanded = user_repr.unsqueeze(1).expand(-1, num_candidates, -1)  # [B, 5, 128]
        user_num_expanded = user_numerical_dense.unsqueeze(1).expand(-1, num_candidates, -1)  # [B, 5, d]

        user_cat_expanded = {
            col: vals.unsqueeze(1).expand(-1, num_candidates)
            for col, vals in user_categorical_input.items()
        }

        # ─────────────────────────────
        # 4. Flatten everything to pairwise [B*5, ...]
        # ─────────────────────────────
        user_repr_flat = user_repr_expanded.reshape(-1, user_repr_expanded.size(-1))
        user_num_flat = user_num_expanded.reshape(-1, user_num_expanded.size(-1))

        user_cat_flat = {
            col: vals.reshape(-1)
            for col, vals in user_cat_expanded.items()
        }

        topic_flat = topic_embed.reshape(-1, topic_embed.size(-1))
        context_flat = context_features.reshape(-1, context_features.size(-1))
        news_repr_flat = news_repr.reshape(-1, news_repr.size(-1))

        # ─────────────────────────────
        # 5. MLP scoring
        # ─────────────────────────────
        score_flat = self.mlp(
            user_repr=user_repr_flat,
            news_repr=news_repr_flat,
            user_numerical_dense=user_num_flat,
            user_categorical_input=user_cat_flat,
            topic_embed=topic_flat,
            context_features=context_flat,
        )

        scores = score_flat.reshape(batch_size, num_candidates)   # [B, 5]
        return scores

##7.1 Initialising Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = NRMSModel().to(device)

##7.2 Initialising Hyperparameters

In [ ]:
# Hyperparameters

learning_rate = 5e-4
batch_size    = 32
num_epochs    = 10

##7.3 Initialising DataLoaders

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=hybrid_collate_fn   # or listwise_collate_fn if same
)
val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=hybrid_variable_collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=hybrid_variable_collate_fn
)

print(f"Train batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")

Train batches: 4424
Val   batches: 31624


##7.4 Ranking Metrics



In [ ]:
def dcg_at_k(r, k):
    r = np.asarray(r, dtype=float)[:k]
    if r.size:
        return np.sum(r / np.log2(np.arange(2, r.size + 2)))
    return 0.0

def ndcg_at_k(r, k):
    dcg_max = dcg_at_k(sorted(r, reverse=True), k)
    if not dcg_max:
        return 0.
    return dcg_at_k(r, k) / dcg_max

def mrr_at_k(y_true, y_score, k=None):
    predictions = sorted(zip(y_score, y_true), key=lambda x: x[0], reverse=True)
    for i, (score, label) in enumerate(predictions):
        if label == 1:
            if k is None or i < k:
                return 1.0 / (i + 1)
            return 0.0
    return 0.0

def average_precision_at_k(y_true, y_score, k=None):
    predictions = sorted(zip(y_score, y_true), key=lambda x: x[0], reverse=True)
    if k is not None:
        predictions = predictions[:k]
    num_relevant, sum_prec = 0, 0.0
    for i, (score, label) in enumerate(predictions):
        if label == 1:
            num_relevant += 1
            sum_prec     += num_relevant / (i + 1)
    return sum_prec / num_relevant if num_relevant else 0.0

def calculate_ranking_metrics(all_labels, all_scores, k_ndcg_5=5, k_ndcg_10=10):
    mrr_scores, map_scores, ndcg_5_scores, ndcg_10_scores = [], [], [], []

    for labels, scores in zip(all_labels, all_scores):
        y_true = np.array(labels)
        y_score = np.array(scores)

        if np.sum(y_true) == 0:
            continue

        mrr_scores.append(mrr_at_k(y_true, y_score))
        map_scores.append(average_precision_at_k(y_true, y_score))

        ranked = sorted(zip(y_score, y_true), key=lambda x: x[0], reverse=True)
        rel_at_rank = [label for _, label in ranked]

        ndcg_5_scores.append(ndcg_at_k(rel_at_rank, k_ndcg_5))
        ndcg_10_scores.append(ndcg_at_k(rel_at_rank, k_ndcg_10))

    return (
        np.mean(mrr_scores) if mrr_scores else 0.0,
        np.mean(map_scores) if map_scores else 0.0,
        np.mean(ndcg_5_scores) if ndcg_5_scores else 0.0,
        np.mean(ndcg_10_scores) if ndcg_10_scores else 0.0,
    )

##7.5 Training

In [ ]:
train_losses_hist = []
train_aucs_hist = []
train_mrrs_hist = []
train_maps_hist = []
train_ndcg5s_hist = []
train_ndcg10s_hist = []

val_losses_hist = []
val_aucs_hist = []
val_mrrs_hist = []
val_maps_hist = []
val_ndcg5s_hist = []
val_ndcg10s_hist = []

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

for epoch in range(num_epochs):
    # =========================
    # TRAIN
    # =========================
    model.train()
    total_train_loss_epoch = 0.0
    total_train_samples_epoch = 0
    running_loss = 0.0

    train_all_labels = []
    train_all_scores = []

    for i, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1} Training")):
        candidate_inputs = {k: v.to(device) for k, v in batch["candidate_inputs"].items()}
        history_inputs = {k: v.to(device) for k, v in batch["history_inputs"].items()}
        user_num = batch["user_numerical"].to(device)
        user_cat = {k: v.to(device) for k, v in batch["user_categorical"].items()}
        topic = batch["topic"].to(device)
        context_features = batch["context_features"].to(device)
        labels = batch["label"].to(device)   # [B], all zeros

        B = user_num.shape[0]

        optimizer.zero_grad()

        scores = model(
            candidate_inputs=candidate_inputs,
            history_inputs=history_inputs,
            user_numerical_dense=user_num,
            user_categorical_input=user_cat,
            topic_embed=topic,
            context_features=context_features,
        )

        loss = F.cross_entropy(scores, labels.long())
        loss.backward()
        optimizer.step()

        total_train_loss_epoch += loss.item() * B
        total_train_samples_epoch += B
        running_loss += loss.item()

        probs = torch.softmax(scores, dim=1)
        train_all_scores.extend(probs.detach().cpu().numpy().tolist())

        num_candidates = scores.shape[1]

        for b in range(B):
            num_candidates = scores.shape[1]
            one_hot = [0] * num_candidates
            one_hot[labels[b].item()] = 1
            train_all_labels.append(one_hot)

        if i % 100 == 99:
            tqdm.write(f"Epoch {epoch+1}, Batch {i+1}, Loss: {running_loss / 100:.4f}")
            running_loss = 0.0

    avg_train_loss = total_train_loss_epoch / total_train_samples_epoch
    train_losses_hist.append(avg_train_loss)

    flat_train_labels = [label for row in train_all_labels for label in row]
    flat_train_scores = [score for row in train_all_scores for score in row]

    if len(np.unique(flat_train_labels)) > 1:
        train_auc = roc_auc_score(flat_train_labels, flat_train_scores)
    else:
        train_auc = 0.0

    train_aucs_hist.append(train_auc)

    train_avg_mrr, train_avg_map, train_avg_ndcg_5, train_avg_ndcg_10 = calculate_ranking_metrics(
        train_all_labels,
        train_all_scores
    )

    train_mrrs_hist.append(train_avg_mrr)
    train_maps_hist.append(train_avg_map)
    train_ndcg5s_hist.append(train_avg_ndcg_5)
    train_ndcg10s_hist.append(train_avg_ndcg_10)

    print(
        f"Epoch {epoch+1} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Train AUC: {train_auc:.4f} | "
        f"MRR: {train_avg_mrr:.4f} | "
        f"MAP: {train_avg_map:.4f} | "
        f"nDCG@5: {train_avg_ndcg_5:.4f} | "
        f"nDCG@10: {train_avg_ndcg_10:.4f}"
    )

    # =========================
    # VALIDATION
    # =========================
    model.eval()
    val_all_labels = []
    val_all_scores = []
    val_loss = 0.0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"):
            candidate_inputs = {k: v.to(device) for k, v in batch["candidate_inputs"].items()}
            history_inputs = {k: v.to(device) for k, v in batch["history_inputs"].items()}
            user_num = batch["user_numerical"].to(device)
            user_cat = {k: v.to(device) for k, v in batch["user_categorical"].items()}
            topic = batch["topic"].to(device)
            context_features = batch["context_features"].to(device)
            labels = batch["labels"].to(device)   # [B, C]

            scores = model(
                candidate_inputs=candidate_inputs,
                history_inputs=history_inputs,
                user_numerical_dense=user_num,
                user_categorical_input=user_cat,
                topic_embed=topic,
                context_features=context_features,
            )

            if labels.sum(dim=1).eq(1).all():
                target_idx = labels.argmax(dim=1)
                val_loss += F.cross_entropy(scores, target_idx).item()

            probs = torch.softmax(scores, dim=1)

            val_all_scores.extend(probs.cpu().numpy().tolist())
            val_all_labels.extend(labels.cpu().numpy().tolist())

    avg_val_loss = val_loss / len(val_loader)
    val_losses_hist.append(avg_val_loss)

    avg_mrr, avg_map, avg_ndcg_5, avg_ndcg_10 = calculate_ranking_metrics(
        val_all_labels,
        val_all_scores
    )

    flat_val_labels = [label for row in val_all_labels for label in row]
    flat_val_scores = [score for row in val_all_scores for score in row]

    if len(np.unique(flat_val_labels)) > 1:
        val_auc = roc_auc_score(flat_val_labels, flat_val_scores)
    else:
        val_auc = 0.0

    val_aucs_hist.append(val_auc)
    val_mrrs_hist.append(avg_mrr)
    val_maps_hist.append(avg_map)
    val_ndcg5s_hist.append(avg_ndcg_5)
    val_ndcg10s_hist.append(avg_ndcg_10)

    print(
        f"Epoch {epoch+1} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val AUC: {val_auc:.4f} | "
        f"MRR: {avg_mrr:.4f} | "
        f"MAP: {avg_map:.4f} | "
        f"nDCG@5: {avg_ndcg_5:.4f} | "
        f"nDCG@10: {avg_ndcg_10:.4f}"
    )

print("Finished Training")

baseline_result = {
    "config": "nrms_hybrid_baseline",
    "removed": "ctr_norm_and_interaction_features",
    "val_auc": round(max(val_aucs_hist), 4),
    "val_mrr": round(max(val_mrrs_hist), 4),
    "val_map": round(max(val_maps_hist), 4),
    "val_ndcg5": round(max(val_ndcg5s_hist), 4),
    "val_ndcg10": round(max(val_ndcg10s_hist), 4),
    "best_epoch": val_ndcg5s_hist.index(max(val_ndcg5s_hist)) + 1,
}
print(f"\nBaseline stored: {baseline_result}")

Epoch 1 Training:   2%|▏         | 107/4424 [00:03<01:13, 58.61it/s]

Epoch 1, Batch 100, Loss: 1.6030


Epoch 1 Training:   5%|▍         | 211/4424 [00:05<01:09, 60.20it/s]

Epoch 1, Batch 200, Loss: 1.5504


Epoch 1 Training:   7%|▋         | 309/4424 [00:07<01:04, 63.33it/s]

...

Epoch 10, Batch 4400, Loss: 1.3204


Epoch 10 Training: 100%|██████████| 4424/4424 [01:15<00:00, 58.95it/s]


Epoch 10 | Train Loss: 1.3412 | Train AUC: 0.7534 | MRR: 0.6547 | MAP: 0.6547 | nDCG@5: 0.7407 | nDCG@10: 0.7407


Epoch 10 Validation: 100%|██████████| 31624/31624 [01:38<00:00, 321.29it/s]


Epoch 10 | Val Loss: 2.0061 | Val AUC: 0.7238 | MRR: 0.3241 | MAP: 0.2990 | nDCG@5: 0.3027 | nDCG@10: 0.3633
Finished Training

Baseline stored: {'config': 'nrms_hybrid_baseline', 'removed': 'ctr_norm_and_interaction_features', 'val_auc': np.float64(0.7246), 'val_mrr': np.float64(0.3278), 'val_map': np.float64(0.3019), 'val_ndcg5': np.float64(0.3075), 'val_ndcg10': np.float64(0.3653), 'best_epoch': 8}


#8. Evaluation

In [ ]:
model.eval()

test_all_labels = []
test_all_scores = []
test_loss = 0.0

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        candidate_inputs = {k: v.to(device) for k, v in batch["candidate_inputs"].items()}
        history_inputs = {k: v.to(device) for k, v in batch["history_inputs"].items()}
        user_num = batch["user_numerical"].to(device)
        user_cat = {k: v.to(device) for k, v in batch["user_categorical"].items()}
        topic = batch["topic"].to(device)
        context_features = batch["context_features"].to(device)
        labels = batch["labels"].to(device)   # [B, C]

        scores = model(
            candidate_inputs=candidate_inputs,
            history_inputs=history_inputs,
            user_numerical_dense=user_num,
            user_categorical_input=user_cat,
            topic_embed=topic,
            context_features=context_features,
        )   # [B, C]

        # optional CE loss for test
        if labels.sum(dim=1).eq(1).all():
            target_idx = labels.argmax(dim=1)
            test_loss += F.cross_entropy(scores, target_idx).item()

        probs = torch.softmax(scores, dim=1)

        test_all_scores.extend(probs.cpu().numpy().tolist())
        test_all_labels.extend(labels.cpu().numpy().tolist())

avg_test_loss = test_loss / len(test_loader)

test_mrr, test_map, test_ndcg5, test_ndcg10 = calculate_ranking_metrics(
    test_all_labels,
    test_all_scores
)

flat_test_labels = [label for row in test_all_labels for label in row]
flat_test_scores = [score for row in test_all_scores for score in row]

if len(np.unique(flat_test_labels)) > 1:
    test_auc = roc_auc_score(flat_test_labels, flat_test_scores)
else:
    test_auc = 0.0

print(
    f"Test Loss: {avg_test_loss:.4f} | "
    f"Test AUC: {test_auc:.4f} | "
    f"Test MRR: {test_mrr:.4f} | "
    f"Test MAP: {test_map:.4f} | "
    f"Test nDCG@5: {test_ndcg5:.4f} | "
    f"Test nDCG@10: {test_ndcg10:.4f}"
)

Testing: 100%|██████████| 30270/30270 [01:34<00:00, 319.51it/s]


Test Loss: 2.0387 | Test AUC: 0.7036 | Test MRR: 0.3013 | Test MAP: 0.2745 | Test nDCG@5: 0.2733 | Test nDCG@10: 0.3337
